# 01 · Part A: build the clean table

This notebook turns the raw PISA 2022 files into **one clean table** for the six systems in our project:
UAE (`ARE`), Saudi Arabia (`SAU`), Qatar (`QAT`), Jordan (`JOR`), Morocco (`MAR`) and Palestine (`PSE`).
Every later notebook (EDA, baselines, neural nets) should start from this table, not from the raw files.

**Part A only applies fixed rules**: steps that don't learn anything from the data, so they are safe to run on all the data at once.
Anything that *learns* from the data (filling in missing values with a median, scaling, percentile clipping) is **Part B**.
Part B has to run inside each training fold, otherwise information from the test rows leaks into training.

| | |
|---|---|
| **Reads** | `data/parquet/CY08MSP_STU_QQQ` (students), `data/parquet/CY08MSP_SCH_QQQ` (schools) |
| **Writes** | `data/clean/pisa6_clean.parquet` (the clean table), `data/clean/columns.json` (which column is what) |
| **Audit files** | `02_data-analysis/outputs/part_a/` (step-by-step row counts, missingness, flagged schools) |
| **Run time** | under a minute |

The ten steps:

1. Keep only the rows we need (six systems)
2. Keep only the columns we need
3. Join school data onto students
4. Understand the missing values (look, don't delete)
5. Check for impossible values
6. Give each column its correct type
7. Fix skewed columns
8. Build the targets (all 10 plausible values)
9. Guard against leakage
10. Add missing indicators, save and freeze

> **AI-use note.** The code in this notebook was generated by an AI assistant (Claude Code, Anthropic, model Claude Opus 5.5) on 17 Nov 2026.
> Following the coursework rule, here are the full prompts:
>
> 1. *"Forget about the branches by fatah and annie. I want to preform the preprocessing, cleanup, and the EDA independent of their work... If we want to perform preprocessing and want to cleanup this data, what are some actions we should take? Assume Im a noob at all of this and need explanations on terminologies and approaches."*
> 2. *"Yes, build Part A"*, answering the question "Should I build the Part A clean-table notebook next?". Part A is the ten steps listed above.
> 3. *"for the notebooks already created, can we improve the var names? python is already hard as its not clear whther a var represents a list or a single value etc."*. The variable names were then changed to follow the naming rules below. The outputs did not change.
>
> The design decisions (the plausibility rule in step 5, merging school types in step 6, the log transform in step 7) are documented where they are made, so the group can review and change them.

## How the variable names work

Python doesn't say what kind of thing a variable holds, so the **names** do. The same rules are used in every notebook in this folder.

| The name... | means it holds | Examples |
|---|---|---|
| is a singular word | **one** value | `system` = `"ARE"`, `feature` = `"ESCS"` |
| is a plural word | a **list** of values | `SYSTEMS` = `["ARE", "SAU", ...]` |
| ends in `_col` | **one** column name | `pv_col` = `"PV1MATH"`, `weight_col` = `"W_FSTUWT"` |
| ends in `_cols` | a **list** of column names | `PV_COLS` = `["PV1MATH", ..., "PV10MATH"]`, `FEATURE_COLS` |
| ends in `_df` | a **table** (a pandas DataFrame) | `students_df`, `system_df` (one system's students) |
| is just `df` | **the main table** of the notebook | |
| looks like `<value>_by_<key>` | a **lookup**: give it a key, get a value (a dict, or a pandas Series) | `FEATURES_BY_GROUP["family"]` gives the 4 family columns |
| starts with `is_` or `has_` | **True/False for each student**, used as a filter | `df[is_complete]` keeps the students with no blanks |
| starts with `n_` / `pct_` / `share_` | a count / a percentage (0-100) / a fraction (0-1) | `n_flagged_students`, `pct_missing_df`, `share_label_flips` |
| ends in `_idx` | a position (0, 1, 2, ...) | `row_idx` |
| ends in `_DIR` | a folder | `CLEAN_DIR`, `FIGURE_DIR` |
| ends in `_COLOR` / `_CMAP` | one colour / a colour scale | `TEXT_COLOR`, `BLUES_CMAP` |
| is in `UPPER_CASE` | a **fixed setting**, set once and never changed | `LEVEL2_CUTOFF`, `SYSTEMS` |
| is `fig`, `ax` or `axes` | a whole figure / one chart panel / several panels | matplotlib's usual names |

Loops use the singular of the list they walk through: `for system in SYSTEMS`, `for feature in FEATURE_COLS`.
Functions are named after what they return (`weighted_quantile()`) or what they do (`log_step()`, `draw_heatmap()`).
Each helper function has **type hints** saying what goes in and what comes out, e.g. `cols: list[str]` (a list of column names) and `-> float` (returns one number).

The column names *inside* the data (`ESCS`, `PV1MATH`, `W_FSTUWT`) are PISA's own. We don't rename them, so they still match the OECD documentation.

## Setup

`ROOT` finds the repository folder wherever you run this from. All the choices you might want to change are constants at the top:
- `LEVEL2_CUTOFF`: the PISA 2022 cut-off for maths Level 2. Students below it are "low achievers".
- `STRATIO_MIN`, `STRATIO_MAX`: the student-teacher ratios we accept as plausible (step 5).
- `PREDICTORS_BY_GROUP`: the 17 predictors from the pitch, in their 5 factor groups.

In [1]:
from pathlib import Path
import json, re
import numpy as np
import pandas as pd

ROOT = next(folder for folder in (Path.cwd(), *Path.cwd().parents) if (folder / "requirements.txt").exists())
PARQUET_DIR = ROOT / "data/parquet"
CLEAN_DIR = ROOT / "data/clean"
OUTPUT_DIR = ROOT / "02_data-analysis/outputs/part_a"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 100,
              "display.float_format", "{:,.2f}".format)

SYSTEMS = ["ARE", "SAU", "QAT", "JOR", "MAR", "PSE"]
LEVEL2_CUTOFF = 420.07             # lower bound of PISA 2022 maths Level 2
STRATIO_MIN, STRATIO_MAX = 2, 50   # plausible student-teacher ratio (step 5)

# The pitch's 17 predictors, in the 5 factor groups used for the ablations.
# A lookup: group name -> list of column names.
PREDICTORS_BY_GROUP = {
    "family":            ["ESCS", "PAREDINT", "HISEI", "HOMEPOS"],
    "maths_attitudes":   ["ANXMAT", "MATHEFF", "MATHEF21"],
    "classroom":         ["DISCLIM", "TEACHSUP"],
    "climate_resources": ["BELONG", "BULLIED", "EDUSHORT", "STAFFSHORT"],
    "school_org":        ["MCLSIZE", "STRATIO", "SCHLTYPE", "SCHSIZE"],
}
SCHOOL_PREDICTOR_COLS = ["EDUSHORT", "STAFFSHORT", "MCLSIZE", "STRATIO", "SCHLTYPE", "SCHSIZE"]   # from the school file
STUDENT_PREDICTOR_COLS = [col for group_cols in PREDICTORS_BY_GROUP.values() for col in group_cols
                          if col not in SCHOOL_PREDICTOR_COLS]                                    # from the student file
PREDICTOR_COLS = STUDENT_PREDICTOR_COLS + SCHOOL_PREDICTOR_COLS                                   # all 17

ID_COLS = ["CNT", "CNTSCHID", "CNTSTUID"]                           # country, school, student
PV_COLS = [f"PV{i}MATH" for i in range(1, 11)]                      # the 10 plausible maths scores
REPLICATE_WEIGHT_COLS = [f"W_FSTURWT{i}" for i in range(1, 81)]     # the 80 replicate weights (for standard errors)
SIDE_QUESTION_COLS = ["REPEAT", "ST273Q06JA", "ST273Q07JA"]         # grade repetition, digital distraction
GENDER_GRADE_COLS = ["ST004D01T", "ST001D01T"]                      # for EDA only, not in the pitch's 17

log_rows = []   # a list with one dict per step; saved as cleaning_log.csv

def log_step(step: str, table_df: pd.DataFrame, what: str) -> None:
    """Record how many rows and columns the table has after a step."""
    log_rows.append({"step": step, "rows": len(table_df), "columns": table_df.shape[1], "what": what})
    print(f"[{step}] {len(table_df):,} rows x {table_df.shape[1]} columns: {what}")

def weighted_mean_over_pvs(table_df: pd.DataFrame, cols: list[str], weight_col: str = "W_FSTUWT") -> float:
    """Weighted mean of each column in cols, then the average of those means.

    cols is the 10 plausible-value columns (or the 10 below-Level-2 columns):
    this is the PISA way to get one number out of 10 plausible values.
    """
    return np.mean([np.average(table_df[col], weights=table_df[weight_col]) for col in cols])

## Steps 1-2: keep only the rows and columns we need

Each **row** is one student. Each **column** is one piece of information about them. The student file has 613,744 rows and 1,278 columns, far more than we need.
We read only the six systems (`filters=`) and only the columns we will use (`columns=`). That keeps memory low and makes it obvious what the analysis depends on.

The columns fall into five kinds:

| Kind | Columns | Used as model input? |
|---|---|---|
| **IDs** | `CNT` country, `CNTSCHID` school, `CNTSTUID` student | No. Needed to split the data by school and to join tables |
| **Features** (predictors) | the 17 in `PREDICTORS_BY_GROUP` | Yes. This is what the model learns from |
| **Targets** | `PV1MATH` ... `PV10MATH` | No. This is what the model predicts |
| **Weights** | `W_FSTUWT`, `SENWT`, `W_FSTURWT1-80` | No. Tells us how many real students each row represents |
| **Side / EDA** | grade repetition, distraction, gender, grade | Only for the side questions and EDA |

`TOTAT` (total number of teachers) is loaded only to explain step 5 and is dropped afterwards.

The school and student IDs are stored as decimals in the source files (e.g. `78400371.0`), so we turn them into whole numbers straight away.

In [2]:
students_df = pd.read_parquet(
    PARQUET_DIR / "CY08MSP_STU_QQQ",
    columns=ID_COLS + STUDENT_PREDICTOR_COLS + PV_COLS + ["W_FSTUWT", "SENWT"]
            + REPLICATE_WEIGHT_COLS + SIDE_QUESTION_COLS + GENDER_GRADE_COLS,
    filters=[("CNT", "in", SYSTEMS)])
schools_df = pd.read_parquet(
    PARQUET_DIR / "CY08MSP_SCH_QQQ",
    columns=["CNT", "CNTSCHID"] + SCHOOL_PREDICTOR_COLS + ["TOTAT"],
    filters=[("CNT", "in", SYSTEMS)])

for table_df, id_cols in [(students_df, ["CNTSCHID", "CNTSTUID"]), (schools_df, ["CNTSCHID"])]:
    for col in id_cols:
        assert (table_df[col] % 1 == 0).all(), f"{col} has non-whole values"
        table_df[col] = table_df[col].astype("int64")
log_step("1-2", students_df, "students: six systems, selected columns")
log_step("1-2", schools_df, "schools: six systems, school predictors + TOTAT")

students_df.groupby("CNT").agg(students=("CNTSTUID", "size"), schools=("CNTSCHID", "nunique")).reindex(SYSTEMS)

[1-2] 61,775 rows x 111 columns: students: six systems, selected columns
[1-2] 1,973 rows x 9 columns: schools: six systems, school predictors + TOTAT


,students,schools
CNT,,
ARE,24600,840
SAU,6928,193
QAT,7676,229
JOR,7799,260
MAR,6867,178
PSE,7905,273


## Step 3: join school data onto students

A **join** (or merge) attaches each student's school information by matching the school ID. It is **many-to-one**: many students share one school.
We match on country *and* school ID, so a school can never be matched to the wrong country.

Checks before trusting the join:
- no duplicate students or schools (a duplicate would silently copy rows),
- the number of students is the same before and after,
- every student found their school.

The **left join** keeps every student even if their school sent no questionnaire. Those students get NaN for the school columns.
The flag `SCH_NODATA` marks them.

From here on, `df` is the main table: one row per student, with their school's columns attached.

In [3]:
assert not students_df.duplicated(["CNT", "CNTSTUID"]).any(), "duplicate students"
assert not schools_df.duplicated(["CNT", "CNTSCHID"]).any(), "duplicate schools"

df = students_df.merge(schools_df, on=["CNT", "CNTSCHID"], how="left", validate="many_to_one", indicator=True)
assert len(df) == len(students_df), "the join changed the number of students"
print("students whose school is not in the school file:", int((df["_merge"] == "left_only").sum()))
df = df.drop(columns="_merge")

df["SCH_NODATA"] = df[SCHOOL_PREDICTOR_COLS].isna().all(axis=1).astype("int8")
log_step("3", df, "school data joined; SCH_NODATA flag added")

(df.groupby("CNT")["SCH_NODATA"]
   .agg(students_without_school_data="sum", pct=lambda no_data_flags: no_data_flags.mean() * 100)
   .reindex(SYSTEMS))

students whose school is not in the school file: 0
[3] 61,775 rows x 119 columns: school data joined; SCH_NODATA flag added


,students_without_school_data,pct
CNT,,
ARE,2424,9.85
SAU,0,0.00
QAT,0,0.00
JOR,0,0.00
MAR,0,0.00
PSE,0,0.00


Every school ID matched. But **2,424 UAE students (9.9% of the UAE sample)** have all six school answers blank.
Their principals most likely didn't return the questionnaire. No other system has this problem.

## Step 4: understand the missing values (look, don't delete)

`NaN` means "no value". The repo's extraction already turned PISA's special codes into NaN:
95 = valid skip, 97 = not applicable, 98 = invalid, 99 = no response.

The first question is *why* values are missing, because that decides what to do:
- **Missing at random**: nothing to do with the student. Dropping those rows would only lose data.
- **Not at random**: missingness goes with the outcome, for example weaker students leaving more answers blank. Dropping those rows would **bias** the data, leaving a sample that looks better than the real population.

In [4]:
# one row per predictor, one column per system
pct_missing_df = df[PREDICTOR_COLS].isna().groupby(df["CNT"]).mean().mul(100).reindex(SYSTEMS).T
pct_missing_df["all six"] = df[PREDICTOR_COLS].isna().mean() * 100
pct_missing_df.round(1).to_csv(OUTPUT_DIR / "missing_by_system.csv")
pct_missing_df

CNT,ARE,SAU,QAT,JOR,MAR,PSE,all six
ESCS,5.66,2.05,6.51,2.08,0.45,2.59,3.94
PAREDINT,5.84,2.17,7.32,2.26,1.82,2.64,4.30
HISEI,15.29,18.42,18.20,13.50,5.45,14.61,14.60
HOMEPOS,4.48,1.40,4.59,0.95,0.22,1.58,2.86
ANXMAT,16.91,16.05,30.39,28.27,37.44,22.05,22.86
MATHEFF,15.82,15.04,28.52,25.84,33.87,19.91,21.10
MATHEF21,16.63,16.17,30.18,27.57,35.53,21.76,22.40
DISCLIM,13.62,12.27,23.32,20.13,23.82,15.91,16.92
TEACHSUP,14.28,12.73,24.82,21.84,26.07,17.03,18.03
BELONG,9.49,6.11,12.96,10.72,5.69,9.29,9.25


In [5]:
is_complete = df[PREDICTOR_COLS].notna().all(axis=1)   # True for students who answered all 17
print(f"Students with all 17 predictors answered: {is_complete.mean():.1%}")
print((is_complete.groupby(df["CNT"]).mean().mul(100).reindex(SYSTEMS).round(1)).to_string(), "\n")

is_missing_attitudes = df[PREDICTORS_BY_GROUP["maths_attitudes"]].isna().all(axis=1)
score_gap = (weighted_mean_over_pvs(df[is_missing_attitudes], PV_COLS)
             - weighted_mean_over_pvs(df[~is_missing_attitudes], PV_COLS))
print(f"Students missing all three maths-attitude answers: {is_missing_attitudes.mean():.1%}")
print(f"Their weighted maths average is {score_gap:+.0f} points compared with everyone else")

Students with all 17 predictors answered: 57.2%
CNT
ARE   56.90
SAU   60.90
QAT   60.30
JOR   61.30
MAR   36.40
PSE   66.10 

Students missing all three maths-attitude answers: 19.5%
Their weighted maths average is -28 points compared with everyone else


**What this tells us**
- Only 57% of students answered all 17 predictors, and just 36% in Morocco.
  Dropping incomplete rows (**complete-case analysis**) would throw away over 40% of the data, and far more from some countries than others.
- One student in five skipped all three maths-attitude questions, and those students score about 28 points lower (weighted).
  So the missing values are **not random**, and *being missing is itself information*.
- The school columns are mostly complete except in the UAE (questionnaires not returned, step 3) and Morocco, where a third of students have no school size or student-teacher ratio.

**What we do**
- We keep every student.
- In step 10 we add a **missing indicator** for each predictor: a 0/1 column such as `ANXMAT_missing` that tells the model "this answer was blank".
- Filling in the blanks (**imputation**) is learned from the data, so it is Part B, inside the training folds.

## Step 5: check for impossible values

An **outlier** is a value far from the rest. Some outliers are real (a very large school). Others are mistakes (a typo in a questionnaire).
We fix only values that are **impossible or implausible**. The rule is written down before any model is trained, and the outcome is never used to choose it.

**The problem.** `STRATIO` (student-teacher ratio) is computed by the OECD as school size ÷ number of teachers, then capped at 1 and 100.
At both ends it produces numbers no secondary school has:
- A **ratio below 2** means fewer than two students per teacher. In these schools the principal usually reported a school size smaller than the teaching staff (e.g. 42 students, 67 teachers).
  The enrolment figure is wrong, so **both `STRATIO` and `SCHSIZE` become NaN**.
- A **ratio above 50** comes from a teacher count that is far too low (e.g. 8,000 students, 2 teachers).
  The teacher count is wrong, so **`STRATIO` becomes NaN** and the school size is kept.

The original values are kept as `STRATIO_raw` and `SCHSIZE_raw`, and every affected school is listed in `outputs/part_a/implausible_schools.csv`.
To change the rule, edit `STRATIO_MIN` / `STRATIO_MAX` at the top.

In [6]:
df["STRATIO_raw"], df["SCHSIZE_raw"] = df["STRATIO"], df["SCHSIZE"]
is_ratio_too_low = df["STRATIO_raw"] < STRATIO_MIN    # True/False per student
is_ratio_too_high = df["STRATIO_raw"] > STRATIO_MAX
df["STRATIO"] = df["STRATIO_raw"].mask(is_ratio_too_low | is_ratio_too_high)
df["SCHSIZE"] = df["SCHSIZE_raw"].mask(is_ratio_too_low)

flagged_schools_df = (
    df[is_ratio_too_low | is_ratio_too_high]
    .groupby(["CNT", "CNTSCHID"])
    .agg(STRATIO_raw=("STRATIO_raw", "first"), SCHSIZE_raw=("SCHSIZE_raw", "first"),
         teachers_TOTAT=("TOTAT", "first"), students_in_sample=("CNTSTUID", "size"))
    .assign(rule=lambda table_df: np.where(table_df["STRATIO_raw"] < STRATIO_MIN,
                                          f"ratio < {STRATIO_MIN}: STRATIO and SCHSIZE set to NaN",
                                          f"ratio > {STRATIO_MAX}: STRATIO set to NaN"))
    .sort_values("STRATIO_raw"))
flagged_schools_df.to_csv(OUTPUT_DIR / "implausible_schools.csv")

n_flagged_students = int((is_ratio_too_low | is_ratio_too_high).sum())
n_low_schools = int((flagged_schools_df["STRATIO_raw"] < STRATIO_MIN).sum())
n_high_schools = int((flagged_schools_df["STRATIO_raw"] > STRATIO_MAX).sum())
print(f"{len(flagged_schools_df)} schools flagged ({n_flagged_students:,} students): "
      f"{n_low_schools} with ratio < {STRATIO_MIN}, {n_high_schools} with ratio > {STRATIO_MAX}")
df = df.drop(columns="TOTAT")
log_step("5", df, "implausible STRATIO / SCHSIZE set to NaN (raw values kept)")
flagged_schools_df

35 schools flagged (1,053 students): 25 with ratio < 2, 10 with ratio > 50
[5] 61,775 rows x 120 columns: implausible STRATIO / SCHSIZE set to NaN (raw values kept)


STRATIO_raw  SCHSIZE_raw  teachers_TOTAT  students_in_sample                                       rule
CNT CNTSCHID                                                                                                         
ARE 78400371         1.00        15.00           72.00                  13  ratio < 2: STRATIO and SCHSIZE set to NaN
    78400725         1.00        16.00           65.00                  12  ratio < 2: STRATIO and SCHSIZE set to NaN
MAR 50400099         1.00         1.00            8.50                   1  ratio < 2: STRATIO and SCHSIZE set to NaN
JOR 40000088         1.00        42.00           67.00                  30  ratio < 2: STRATIO and SCHSIZE set to NaN
SAU 68200124         1.00        14.00           33.00                  12  ratio < 2: STRATIO and SCHSIZE set to NaN
    68200142         1.00        10.00           21.50                   7  ratio < 2: STRATIO and SCHSIZE set to NaN
QAT 63400056         1.00        48.00           80.00                  44  ratio < 2: STRATIO and SCHSIZE set to NaN
SAU 68200044         1.00        25.00           29.00                  23  ratio < 2: STRATIO and SCHSIZE set to NaN
    68200153         1.00        53.00           59.00                  41  ratio < 2: STRATIO and SCHSIZE set to NaN
PSE 27500071         1.06        35.00           33.00                  35  ratio < 2: STRATIO and SCHSIZE set to NaN
SAU 68200136         1.19        22.00           18.50                  12  ratio < 2: STRATIO and SCHSIZE set to NaN
ARE 78400548         1.38        55.00           40.00                  41  ratio < 2: STRATIO and SCHSIZE set to NaN
SAU 68200191         1.39        53.00           38.00                  42  ratio < 2: STRATIO and SCHSIZE set to NaN
    68200070         1.43        50.00           35.00                  36  ratio < 2: STRATIO and SCHSIZE set to NaN
    68200082         1.43        53.00           37.00                  42  ratio < 2: STRATIO and SCHSIZE set to NaN
PSE 27500206         1.46        35.00           24.00                  33  ratio < 2: STRATIO and SCHSIZE set to NaN
SAU 68200100         1.46        35.00           24.00                  13  ratio < 2: STRATIO and SCHSIZE set to NaN
    68200147         1.50        63.00           42.00                  34  ratio < 2: STRATIO and SCHSIZE set to NaN
    68200135         1.58        52.00           33.00                  41  ratio < 2: STRATIO and SCHSIZE set to NaN
ARE 78400280         1.62        60.00           37.00                  28  ratio < 2: STRATIO and SCHSIZE set to NaN
SAU 68200071         1.66        58.00           35.00                  35  ratio < 2: STRATIO and SCHSIZE set to NaN
    68200117         1.71        29.00           17.00                  20  ratio < 2: STRATIO and SCHSIZE set to NaN
ARE 78400716         1.77        53.00           30.00                  40  ratio < 2: STRATIO and SCHSIZE set to NaN
MAR 50400173         1.86       119.00           64.00                  42  ratio < 2: STRATIO and SCHSIZE set to NaN
ARE 78400784         1.91        86.00           45.00                  82  ratio < 2: STRATIO and SCHSIZE set to NaN
    78400127        50.51     6,314.00          125.00                  40             ratio > 50: STRATIO set to NaN
MAR 50400141        50.56     1,618.00           32.00                  38             ratio > 50: STRATIO set to NaN
JOR 40000236        51.18       870.00           17.00                  22             ratio > 50: STRATIO set to NaN
ARE 78400082        53.06     6,314.00          119.00                  39             ratio > 50: STRATIO set to NaN
MAR 50400069        53.86       781.00           14.50                  42             ratio > 50: STRATIO set to NaN
SAU 68200161        59.11       532.00            9.00                  13             ratio > 50: STRATIO set to NaN
QAT 63400048        65.00       195.00            3.00                   2             ratio > 50: STRATIO set to NaN
SA

**What we deliberately don't do here.** Some indices have long tails: home possessions (`HOMEPOS`) runs from about -9 to +9 while most students sit between -4 and +2.
Those values are real, just extreme. Linear models and neural nets can be pulled around by them, so they are usually **clipped** at the 1st and 99th percentile (**winsorising**).
But a percentile is *learned from the data*, so clipping belongs in Part B, inside the training folds. Tree models don't need it at all.

In [7]:
df[PREDICTOR_COLS].describe(percentiles=[.01, .5, .99]).T[["count", "min", "1%", "50%", "99%", "max"]]

,count,min,1%,50%,99%,max
ESCS,"59,343.00",-6.39,-3.74,-0.02,1.58,5.40
PAREDINT,"59,116.00",3.00,3.00,16.00,16.00,16.00
HISEI,"52,758.00",11.01,17.00,65.01,88.70,88.96
HOMEPOS,"60,011.00",-9.23,-4.06,-0.74,2.12,9.30
ANXMAT,"47,652.00",-2.39,-2.39,0.19,2.63,2.63
MATHEFF,"48,738.00",-3.51,-3.45,-0.43,2.27,2.36
MATHEF21,"47,938.00",-2.49,-2.28,0.20,2.59,2.91
DISCLIM,"51,320.00",-2.55,-2.49,-0.08,1.85,1.85
TEACHSUP,"50,635.00",-2.91,-2.91,0.15,1.56,1.56
BELONG,"56,061.00",-3.32,-1.91,-0.34,2.75,2.78


## Step 6: give each column its correct type

Numbers in a table don't always mean "amount":
- **Numeric**: the number is a real quantity. Most of our predictors are PISA **indices**, scores built from several questionnaire items and centred near 0 for the OECD average.
- **Categorical**: the number is just a label. `SCHLTYPE` uses 1 = private independent, 2 = private government-dependent, 3 = public. "3" is not three times "1", so feeding the raw code to a linear model would be wrong.
- **Ordinal**: ordered groups. `MCLSIZE` (maths class size) is stored as the midpoint of each band (13 = "15 or fewer", 18 = "16-20", ... 53 = "more than 50"), so treating it as a number is fine.

**Rare category.** Only 427 students (in three systems) are in type 2 schools, too few for a model to learn a separate effect.
We merge types 1 and 2 into one binary feature, **`SCHL_PRIVATE`** (1 = private, 0 = public).
`SCHLTYPE` itself stays in the table for EDA.

In [8]:
school_type_name_by_code = {1: "1 private indep.", 2: "2 private gov-dep.", 3: "3 public"}
print(pd.crosstab(df["CNT"], df["SCHLTYPE"].map(school_type_name_by_code), dropna=False)
        .reindex(SYSTEMS).to_string(), "\n")

df["SCHL_PRIVATE"] = df["SCHLTYPE"].map({1.0: 1.0, 2.0: 1.0, 3.0: 0.0})
assert df["SCHLTYPE"].isna().equals(df["SCHL_PRIVATE"].isna())

assert set(df["REPEAT"].dropna().unique()) <= {0.0, 1.0}
print("MCLSIZE codes:", [int(size_code) for size_code in sorted(df["MCLSIZE"].dropna().unique())])
log_step("6", df, "SCHL_PRIVATE added")

SCHLTYPE  1 private indep.  2 private gov-dep.  3 public   NaN
CNT                                                           
ARE                  14032                 205      7900  2463
SAU                   1244                   0      5684     0
QAT                   3598                 171      3907     0
JOR                   1308                   0      6491     0
MAR                    595                   0      6272     0
PSE                    376                  51      7478     0 



MCLSIZE codes: [13, 18, 23, 28, 33, 38, 43, 48, 53]
[6] 61,775 rows x 121 columns: SCHL_PRIVATE added


## Step 7: fix skewed columns

A **skewed** column has most values bunched together and a long tail on one side. **Skewness** measures this: 0 is symmetric, and anything beyond about ±1 is strongly skewed.
A long tail lets a few extreme values dominate linear models and neural nets.

`SCHSIZE` (number of students in the school) is a positive count with a very long right tail. Taking the **log** squeezes the tail,
so the gap between 100 and 1,000 students counts the same as the gap between 1,000 and 10,000. The feature we model with is **`SCHSIZE_log`**.

In [9]:
df["SCHSIZE_log"] = np.log(df["SCHSIZE"])
numeric_cols = [col for col in PREDICTOR_COLS if col != "SCHLTYPE"] + ["SCHSIZE_log"]
skewness_df = df[numeric_cols].skew().rename("skewness").to_frame()
skewness_df["strongly skewed"] = skewness_df["skewness"].abs() > 1
log_step("7", df, "SCHSIZE_log added")
skewness_df.round(2)

[7] 61,775 rows x 122 columns: SCHSIZE_log added


,skewness,strongly skewed
ESCS,-0.81,False
PAREDINT,-1.90,True
HISEI,-0.42,False
HOMEPOS,0.12,False
ANXMAT,-0.13,False
MATHEFF,-0.06,False
MATHEF21,-0.01,False
DISCLIM,-0.12,False
TEACHSUP,-0.63,False
BELONG,1.01,True


`SCHSIZE` drops from a skewness of 3.1 to about 0 after the log. Three other columns are just past ±1, and we leave them alone:
- **`PAREDINT`** (parents' years of schooling) is skewed to the *left* because at least half the students are at the top of the scale (16 years, a university degree).
  That is a **ceiling**, a real limit of the scale. A log can't fix it and wouldn't make sense.
- **`BELONG`** and **`STRATIO`** are only mildly skewed (about 1.0-1.1). Tree models don't care, and the in-fold clipping and scaling in Part B handles the rest.

(`SCHLTYPE` is left out of the table: skewness means nothing for a category.)

## Step 8: build the targets

The **target** (or label) is what the model predicts. PISA doesn't give one maths score per student, it gives **10 plausible values** (`PV1MATH` ... `PV10MATH`).
Each student answered only part of the test, so their true score is uncertain, and the 10 values are 10 likely draws of it.

The OECD rule is: run the analysis once per plausible value, then combine the 10 results (**Rubin's rules**). Never average the 10 into a single score.

That gives two targets for each plausible value `k`:
- **Regression**: `PVkMATH`, the score itself.
- **Classification**: `BELOW_L2_PVk` = 1 if `PVkMATH` < 420.07 (below Level 2), else 0.

In [10]:
BELOW_L2_COLS = [f"BELOW_L2_PV{i}" for i in range(1, 11)]
for pv_col, below_col in zip(PV_COLS, BELOW_L2_COLS):   # PV1MATH -> BELOW_L2_PV1, and so on
    df[below_col] = (df[pv_col] < LEVEL2_CUTOFF).astype("int8")
assert df[PV_COLS].notna().all().all(), "missing maths scores"

print("Three students' 10 plausible values:")
print(df[PV_COLS].head(3).round(0).to_string(), "\n")
share_label_flips = df[BELOW_L2_COLS].nunique(axis=1).gt(1).mean()
print(f"Students whose below-Level-2 label changes across the 10 values: {share_label_flips:.1%}\n")

summary_df = pd.DataFrame({
    "maths_mean": df.groupby("CNT").apply(lambda system_df: weighted_mean_over_pvs(system_df, PV_COLS)),
    "pct_below_L2": df.groupby("CNT").apply(lambda system_df: weighted_mean_over_pvs(system_df, BELOW_L2_COLS) * 100),
    "share_of_pooled_weight_%": df.groupby("CNT")["W_FSTUWT"].sum() / df["W_FSTUWT"].sum() * 100,
}).reindex(SYSTEMS)
print(summary_df.round(1).to_string(), "\n")
print(f"Pooled % below Level 2 with W_FSTUWT (population weight): {weighted_mean_over_pvs(df, BELOW_L2_COLS) * 100:.1f}")
print(f"Pooled % below Level 2 with SENWT (each system equal):     {weighted_mean_over_pvs(df, BELOW_L2_COLS, 'SENWT') * 100:.1f}")
log_step("8", df, "10 classification targets added")

Three students' 10 plausible values:
   PV1MATH  PV2MATH  PV3MATH  PV4MATH  PV5MATH  PV6MATH  PV7MATH  PV8MATH  PV9MATH  PV10MATH
0   419.00   387.00   395.00   456.00   436.00   448.00   437.00   441.00   419.00    413.00
1   300.00   363.00   385.00   305.00   370.00   328.00   345.00   361.00   296.00    334.00
2   292.00   240.00   203.00   252.00   235.00   274.00   262.00   269.00   212.00    233.00 



Students whose below-Level-2 label changes across the 10 values: 32.2%

     maths_mean  pct_below_L2  share_of_pooled_weight_%
CNT                                                    
ARE      431.10         49.00                      5.60
SAU      388.80         70.00                     29.30
QAT      414.10         56.50                      1.70
JOR      361.20         82.80                     13.30
MAR      364.80         81.60                     42.00
PSE      365.70         79.90                      8.20 

Pooled % below Level 2 with W_FSTUWT (population weight): 76.0
Pooled % below Level 2 with SENWT (each system equal):     70.0
[8] 61,775 rows x 132 columns: 10 classification targets added


**Things to notice**
- For about a third of students the label flips between plausible values. That is why we use all 10.
- **The positive class is the majority.** About 76% of students (population-weighted) are below Level 2. A model that says "everyone is below" scores 76% accuracy, so accuracy alone is a poor metric.
- **The weight choice changes the answer.** With `W_FSTUWT`, Morocco and Saudi Arabia make up about 71% of the pooled data. With `SENWT` every system counts equally. This is a team decision for Part B; both weights are in the table.

## Step 9: guard against leakage

**Leakage** means the model sees information it wouldn't have in real use, usually something that is secretly a copy of the answer. It gives great scores that mean nothing.
For us, the leaky columns are:
- **reading and science scores** (`PVxREAD`, `PVxSCIE`): they come from the same test session and correlate very strongly with maths;
- **maths sub-scale scores** (`PVxMCCR`, `PVxMPFS`, ...): they *are* maths scores, split by topic;
- **test answers and timings** (`STU_COG`, `STU_TIM` files): the raw material the scores are computed from;
- **IDs and weights**: they describe the sample, not the student.

We never loaded the first three. This cell checks that, and checks that no ID, weight or target ends up in the feature list.

In [11]:
# The model uses SCHL_PRIVATE instead of SCHLTYPE (step 6) and SCHSIZE_log instead of SCHSIZE (step 7).
model_col_by_raw_col = {"SCHLTYPE": "SCHL_PRIVATE", "SCHSIZE": "SCHSIZE_log"}
FEATURES_BY_GROUP = {group: [model_col_by_raw_col.get(col, col) for col in group_cols]
                     for group, group_cols in PREDICTORS_BY_GROUP.items()}
FEATURE_COLS = [col for group_cols in FEATURES_BY_GROUP.values() for col in group_cols]

LEAKY_COL_PATTERN = re.compile(r"^PV\d+(READ|SCIE|MC[A-Z]{2}|MP[A-Z]{2})$")
assert not [col for col in df.columns if LEAKY_COL_PATTERN.match(col)], "leaky score columns present"
non_feature_cols = set(ID_COLS + PV_COLS + BELOW_L2_COLS + ["W_FSTUWT", "SENWT"] + REPLICATE_WEIGHT_COLS)
assert not non_feature_cols & set(FEATURE_COLS), "an ID, target or weight is in the feature list"
assert len(FEATURE_COLS) == 17
print("Leakage checks passed. The 17 model features:")
for group, group_cols in FEATURES_BY_GROUP.items():
    print(f"  {group:<18} {group_cols}")

Leakage checks passed. The 17 model features:
  family             ['ESCS', 'PAREDINT', 'HISEI', 'HOMEPOS']
  maths_attitudes    ['ANXMAT', 'MATHEFF', 'MATHEF21']
  classroom          ['DISCLIM', 'TEACHSUP']
  climate_resources  ['BELONG', 'BULLIED', 'EDUSHORT', 'STAFFSHORT']
  school_org         ['MCLSIZE', 'STRATIO', 'SCHL_PRIVATE', 'SCHSIZE_log']


## Step 10: add missing indicators, save and freeze

**Missing indicators.** For each of the 17 features we add `<feature>_missing` (1 = blank, 0 = answered).
They are created *after* step 5, so they also mark the values we set to NaN there.
`columns.json` lists them by factor group, so an ablation that removes a group also removes its indicators.
Otherwise the "removed" group would sneak back in through its missing flags.

**Side-question variables.** `DISTRACTED_OWN` / `DISTRACTED_OTHERS` = 1 if the student says digital devices distract them (or others) in *most or every* maths lesson, matching the pitch's definition.
`REPEAT` (repeated a grade: 0/1) is kept as it is.

**Freeze.** We save one Parquet file plus `columns.json`, which says which column is an ID, target, feature, weight, side-question or audit column.
Then we read the file back and check it is identical. From now on, everyone uses this file. If a cleaning rule changes, re-run this notebook; don't patch the table by hand.

In [12]:
MISSING_FLAGS_BY_GROUP = {group: [f"{col}_missing" for col in group_cols]
                          for group, group_cols in FEATURES_BY_GROUP.items()}
for feature in FEATURE_COLS:
    df[f"{feature}_missing"] = df[feature].isna().astype("int8")

for new_col, item_col in [("DISTRACTED_OWN", "ST273Q06JA"), ("DISTRACTED_OTHERS", "ST273Q07JA")]:
    df[new_col] = (df[item_col] <= 2).astype("float64").where(df[item_col].notna())

# What each column is for. Saved as columns.json so later notebooks can look it up.
column_roles = {
    "ids": ID_COLS,
    "targets": {"regression": PV_COLS, "classification": BELOW_L2_COLS, "level2_cutoff": LEVEL2_CUTOFF},
    "feature_groups": FEATURES_BY_GROUP,
    "missing_indicators": MISSING_FLAGS_BY_GROUP,
    "flags": ["SCH_NODATA"],
    "weights": {"final": "W_FSTUWT", "senate": "SENWT", "replicates": REPLICATE_WEIGHT_COLS},
    "side_questions": SIDE_QUESTION_COLS + ["DISTRACTED_OWN", "DISTRACTED_OTHERS"],
    "eda_only": GENDER_GRADE_COLS + ["SCHLTYPE", "SCHSIZE"],
    "audit_only": ["STRATIO_raw", "SCHSIZE_raw"],
}
missing_flag_cols = [col for group_cols in MISSING_FLAGS_BY_GROUP.values() for col in group_cols]
ordered_cols = (ID_COLS + PV_COLS + BELOW_L2_COLS + FEATURE_COLS + missing_flag_cols + ["SCH_NODATA"]
                + ["W_FSTUWT", "SENWT"] + REPLICATE_WEIGHT_COLS
                + column_roles["side_questions"] + column_roles["eda_only"] + column_roles["audit_only"])
assert sorted(ordered_cols) == sorted(df.columns), set(df.columns) ^ set(ordered_cols)
df = df[ordered_cols]
log_step("10", df, "missing indicators + side-question flags added; final column order")

clean_path = CLEAN_DIR / "pisa6_clean.parquet"
df.to_parquet(clean_path, index=False)
(CLEAN_DIR / "columns.json").write_text(json.dumps(column_roles, indent=2) + "\n")
pd.DataFrame(log_rows).to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False)

reloaded_df = pd.read_parquet(clean_path)
pd.testing.assert_frame_equal(reloaded_df, df)
print(f"\nSaved and re-read {clean_path.relative_to(ROOT)}: {reloaded_df.shape[0]:,} rows x {reloaded_df.shape[1]} columns, "
      f"{clean_path.stat().st_size / 1e6:.1f} MB")
pd.DataFrame(log_rows)

[10] 61,775 rows x 151 columns: missing indicators + side-question flags added; final column order



Saved and re-read data\clean\pisa6_clean.parquet: 61,775 rows x 151 columns, 18.0 MB


,step,rows,columns,what
0,1-2,61775,111,"students: six systems, selected columns"
1,1-2,1973,9,"schools: six systems, school predictors + TOTAT"
2,3,61775,119,school data joined; SCH_NODATA flag added
3,5,61775,120,implausible STRATIO / SCHSIZE set to NaN (raw ...
4,6,61775,121,SCHL_PRIVATE added
5,7,61775,122,SCHSIZE_log added
6,8,61775,132,10 classification targets added
7,10,61775,151,missing indicators + side-question flags added...


## How to use the clean table in your own notebook

```python
from pathlib import Path
import json
import pandas as pd

ROOT = next(folder for folder in (Path.cwd(), *Path.cwd().parents) if (folder / "requirements.txt").exists())
df = pd.read_parquet(ROOT / "data/clean/pisa6_clean.parquet")
column_roles = json.loads((ROOT / "data/clean/columns.json").read_text())

feature_cols = [col for group_cols in column_roles["feature_groups"].values() for col in group_cols]  # the 17 model inputs
pv_cols = column_roles["targets"]["regression"]              # PV1MATH ... PV10MATH
below_l2_cols = column_roles["targets"]["classification"]    # BELOW_L2_PV1 ... BELOW_L2_PV10

X = df[feature_cols]                                  # model inputs: a table
y_score = df[pv_cols[0]]                              # regression target for PV1 (repeat for all 10)
y_below_l2 = df[below_l2_cols[0]]                     # classification target for PV1
sample_weight = df[column_roles["weights"]["final"]]  # W_FSTUWT
school_ids = df["CNTSCHID"]                           # for school-grouped folds
```

`X` and `y` are the usual machine-learning names: `X` (capital, because it is a table) holds the inputs and `y` holds the target.

**Still to do in Part B (inside each training fold):** impute missing values, clip extreme values, scale, one-hot encode, and pick the weight.